#### Code to run the FCNN model on TCGA bulk data and get the CV accuracy  values

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, zscore
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns 
import random

In [2]:
tf.keras.backend.clear_session()

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
tcga_features = pd.read_csv("/data/kumarr17/common_features_ver3.txt", header=None)
cyt_features = pd.read_csv("/data/kumarr17/common_cyt_nichenet_gulden_tcga.txt", header=None)
tcga_features_list=list(tcga_features[0])
cyt_features_list=list(cyt_features[0])

In [4]:
df_tcga=pd.read_csv("/data/kumarr17/merged_sample_mtx/TCGA_Merged_mRNA_Expression.tsv", index_col=0)

In [5]:
df_tcga_features=df_tcga[tcga_features_list]
df_tcga_cyt_col=df_tcga[cyt_features_list]
filtered_model_feature_list=tcga_features_list

In [6]:
df_tcga_features
df_tcga_cyt_col

,INHA,FGF7,BMP6,IL25,CCL21,LIPH,SFTPA1,FGF23,ADCYAP1,MIA,...,ZP3,PSEN1,B2M,LTF,GALP,ANGPT2,CALCB,CSH1,CNTN2,NODAL
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
TCGA-AB-2803-03,5.1440,15.4321,18.9815,0.0,0.0000,14.4033,2.0576,1.0288,0.0000,3.3230,...,136.6564,2614.1975,73139.9074,858.0247,0.0000,111.1111,0.0000,0.0,5.1440,10.2881
TCGA-AB-2805-03,0.0000,0.6693,11.5797,0.0,0.0000,7.3628,0.0000,0.0000,0.0000,4.9933,...,65.1339,4286.4793,33497.9920,107.0950,0.6693,109.7724,0.0000,0.0,8.7015,21.4190
TCGA-AB-2806-03,0.0000,17.1865,29.5544,0.0,0.6365,15.9134,2.5461,0.6365,0.0000,7.1356,...,145.5697,2728.8351,23352.6416,26.7346,0.0000,96.7537,1.9096,0.0,5.7288,10.1846
TCGA-AB-2807-03,3.9722,97.3188,83.1480,0.0,0.0000,2.9791,14.8957,0.9930,16.8818,1.0427,...,120.0497,2016.8818,41025.8193,39.7219,0.0000,178.7488,0.9930,0.0,7.9444,30.7845
TCGA-AB-2808-03,0.0000,14.1398,20.7306,0.0,0.0000,7.0699,0.0000,0.0000,0.7855,2.4666,...,49.8272,2492.5373,38730.5577,14278.0833,1.5711,225.4517,0.0000,0.0,3.1422,11.7832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-VQ-AA6I-01,0.9621,112.0799,91.1575,0.0,656.3646,4186.6394,0.4810,0.0000,8.1775,30.3049,...,67.8468,2739.4624,59258.0121,9938.3080,0.0000,238.8311,0.0000,0.0,1.2026,3.6077
TCGA-VQ-AA6J-01,11.9710,121.1468,58.0308,0.0,455.1386,1424.3132,0.0000,0.0000,0.0000,59.1608,...,84.6376,1947.6866,66351.5892,1588.3163,0.0000,523.8523,0.2394,0.0,3.8307,6.2249
TCGA-VQ-AA6K-01,127.5618,393.6396,122.3887,0.0,518.3746,1754.7703,0.0000,1.0601,71.7314,25.0106,...,36.3145,2693.2862,52730.3887,12.0141,0.0000,291.8728,0.0000,0.0,2.4735,9.1873


In [ ]:
# -----------------------------
# speed settings
# -----------------------------
tf.keras.backend.clear_session()

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

cyt_tmp = cyt_of_interest
corr_new_tmp = []

# features
X_raw = df_tcga_features.values.astype("float32")

# optional: row-wise z-score, faster than apply(zscore, axis=1)
X_raw = (X_raw - X_raw.mean(axis=1, keepdims=True)) / (
    X_raw.std(axis=1, keepdims=True) + 1e-8
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def build_fcnn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mean_squared_error",
        metrics=["mae"]
    )
    return model

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

for cytokine_to_check in cyt_tmp:
    
    print(f"\n==============================")
    print(f"Cytokine: {cytokine_to_check}")
    print(f"==============================")
    
    y = df_tcga_cyt_col[[cytokine_to_check]].values.astype("float32")
    
    corr_lt = []
    
    for fold, (train_index, val_index) in enumerate(kf.split(X_raw)):
        print(f"Fold {fold+1}/5")
        
        X_train_raw = X_raw[train_index]
        X_val_raw   = X_raw[val_index]
        y_train     = y[train_index]
        y_val       = y[val_index]
        
        # scale using training fold only
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw).astype("float32")
        X_val   = scaler.transform(X_val_raw).astype("float32")
        
        tf.keras.backend.clear_session()
        model = build_fcnn(X_train.shape[1])
        
        model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=128,      # larger batch = faster
            verbose=0,
            callbacks=[early_stop]
        )
        
        y_pred = model.predict(X_val, verbose=0).ravel()
        y_true = y_val.ravel()
        
        corr, p_value = pearsonr(y_true, y_pred)
        corr_lt.append(corr)
    
    mean_corr = np.mean(corr_lt)
    corr_new_tmp.append(mean_corr)
    
    print("Mean CV Pearson:", mean_corr)

CV_df = pd.DataFrame(
    {"CV": corr_new_tmp},
    index=cyt_tmp
)


Cytokine: CXCL1
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.6449784902339821

Cytokine: CNTN4
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.7928761639830336

Cytokine: INSL3
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.5022776522675627

Cytokine: IFNA17
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.013954113907513065

Cytokine: FGF23
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.35957605265806475

Cytokine: PROK1
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.42618091107188627

Cytokine: IFNA2
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.1606086127708773

Cytokine: EGF
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.8086189752971403

Cytokine: FGF10
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.7278889211647704

Cytokine: CXCL9
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.838875678272314

Cytokine: OLFM2
F

### Plots figure 1

In [10]:
TCGA_bulk_cv_df=pd.read_csv("/data/kumarr17/CV_df_7k_features.csv",index_col='Unnamed: 0') 

In [59]:
TCGA_bulk_cv_df.columns=['Tumor_bulk']

In [60]:
import glob, os
os.chdir("/data/kumarr17/supplemetary_table_cyt_manuscript/")
cv_data_df=[] 
for i in glob.glob("*"+'.csv'):
    cv_data_df.append(i)
os.chdir("/data/kumarr17/supplemetary_table_cyt_manuscript/") 

In [62]:
list_df=[]
for i in range(11):
    cv_df=pd.read_csv("/data/kumarr17/supplemetary_table_cyt_manuscript/"+cv_data_df[i],index_col='Unnamed: 0')
    cv_df.columns=[cv_data_df[i].split('_')[0]]
    list_df.append(cv_df)

In [63]:
list_df.append(TCGA_bulk_cv_df)

In [64]:
pd.concat(list_df,axis=1,join="inner")

,CD14,CD19,CD4,CD56,CD8,Cancer,Endothelial,Eos,Fibroblast,Neu,Treg,Tumor_bulk
CXCL1,0.438498,0.381382,0.376647,0.411023,0.418974,0.633333,0.476089,0.420532,0.486616,0.503582,0.513833,0.662190
CNTN4,0.630547,0.597611,0.538652,0.602184,0.615322,0.720406,0.654172,0.572835,0.643098,0.761030,0.529050,0.796657
INSL3,0.563998,0.544576,0.442005,0.531812,0.549001,0.648755,0.549444,0.507482,0.572344,0.611658,0.573001,0.550836
IFNA17,0.003732,-0.005069,0.007477,0.021587,0.001448,-0.010960,0.008556,0.019283,0.015790,-0.010947,0.003676,0.006075
FGF23,0.252499,0.205535,0.142697,0.204247,0.197996,0.249660,0.311096,0.123487,0.243950,0.230686,0.188152,0.384862
...,...,...,...,...,...,...,...,...,...,...,...,...
CSF2,0.478595,0.450144,0.351818,0.416424,0.429633,0.647282,0.483677,0.418446,0.510712,0.569304,0.506339,0.656440
DKK1,0.338646,0.327904,0.224606,0.293723,0.260388,0.471964,0.306013,0.291732,0.356405,0.347681,0.340554,0.488177
JAG1,0.685462,0.663242,0.606806,0.687151,0.661410,0.778871,0.688921,0.659443,0.709183,0.749087,0.657496,0.817610
LAMA3,0.747102,0.722059,0.683768,0.738451,0.737787,0.861918,0.774994,0.731443,0.778583,0.805091,0.807821,0.872237
